In [ ]:
7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA

In [3]:
from telegram import Update
from telegram.ext import Application, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

# Replace with your bot token
BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'

async def get_chat_id(update: Update, context: CallbackContext) -> None:
    # Check if the update has a message
    if update.message:
        chat_id = update.effective_chat.id
        print(f"Group Chat ID: {chat_id}")
        await update.message.reply_text(f"Group Chat ID: {chat_id}")

async def run_bot():
    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Message handler to respond with the chat ID when a message is sent in a group
    app.add_handler(MessageHandler(filters.TEXT & filters.ChatType.GROUPS, get_chat_id))

    # Run the bot until the notebook is interrupted
    await app.run_polling()

# Run the bot
await run_bot()


INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getMe "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/deleteWebhook "HTTP/1.1 200 OK"
INFO:telegram.ext.Application:Application started
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTT

Group Chat ID: -1001983252802


INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:telegram.ext.Application:Application is stopping. This might take a moment.
INFO:telegram.ext.Application:Application.stop() complete


RuntimeError: Cannot close a running event loop

In [1]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio
import hashlib
import hmac
import requests
import time
import logging
import uuid

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

# Replace with your bot token
BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'
# Replace with your group chat ID (use a negative number for groups)
# GROUP_CHAT_ID = -4553279527  # Update this with your group's chat ID-1001983252802
GROUP_CHAT_ID = -1001983252802  # Update this with your group's chat ID-1001983252802


# Sumsub API credentials
SUMSUB_APP_TOKEN = "prd:WOCxi7YZ65cLyCJqRNS69i5d.5umRA9KrRyyM5N8GNj6IeJuwQ0xKiOF0"  # Replace with your actual secret key
SUMSUB_SECRET_KEY = "bOcz37g6kKp6RTAf3mObbJkqSRrELvu3"  # Replace with your actual app token
SUMSUB_BASE_URL = "https://api.sumsub.com"  # Make sure to use https and correct URL (sandbox or production)
REQUEST_TIMEOUT = 60

async def start(update: Update, context: CallbackContext) -> None:
    if update.message:
        user_id = update.message.from_user.id
        chat_member = await context.bot.get_chat_member(GROUP_CHAT_ID, user_id)

        # Check if the user is a member of the group
        if chat_member.status in ['member', 'administrator', 'creator']:
            await update.message.reply_text("Welcome, you are a member of the group!")
        else:
            await update.message.reply_text("Sorry, this bot is only for members of the group.")

async def handle_message(update: Update, context: CallbackContext) -> None:
    # Check if update.message exists
    if update.message and update.message.from_user:
        user_id = update.message.from_user.id
        chat_member = await context.bot.get_chat_member(GROUP_CHAT_ID, user_id)

        if chat_member.status in ['member', 'administrator', 'creator']:
            # Check if the message is a command for KYC or POA
            message_text = update.message.text.lower()

            if '@sumsub_pre_bot kyc' in message_text:
                # Generate KYC link
                websdk_link = generate_websdk_link('basic-kyc-level', str(uuid.uuid4()))
                if websdk_link:
                    await update.message.reply_text(f"Here is your KYC link: {websdk_link}")
                else:
                    await update.message.reply_text("Sorry, there was an error generating your KYC link.")
            
            elif '@sumsub_pre_bot poa' in message_text:
                # Generate POA link
                websdk_link = generate_websdk_link('POA', str(uuid.uuid4()))
                if websdk_link:
                    await update.message.reply_text(f"Here is your POA link: {websdk_link}")
                else:
                    await update.message.reply_text("Sorry, there was an error generating your POA link.")

            # anything else with@sumsub_pre_bot but not kyc or poa
            elif '@sumsub_pre_bot' in message_text:
                await update.message.reply_text("I'm sorry, I don't understand that command. Please use '@sumsub_pre_bot kyc' or '@sumsub_pre_bot poa'.")
        else:
            await update.message.reply_text("Sorry, this bot is only for members of the group.")

def generate_websdk_link(level_name, external_user_id, ttl_in_secs=1800, lang='en'):
    # Endpoint to create a WebSDK link
    url = f"{SUMSUB_BASE_URL}/resources/sdkIntegrations/levels/{level_name}/websdkLink"
    params = {
        'ttlInSecs': ttl_in_secs,
        'externalUserId': external_user_id,
        'lang': lang
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }
    try:
        response = sign_request(requests.Request('POST', url, params=params, headers=headers, data="{}"))
        session = requests.Session()
        resp = session.send(response, timeout=REQUEST_TIMEOUT)
        logging.info(f"Generate WebSDK Link Response: {resp.status_code} - {resp.text}")
        resp.raise_for_status()  # Raise an error for bad status codes
        websdk_url = resp.json().get('url')
        return websdk_url
    except Exception as e:
        logging.error(f"Error generating WebSDK link: {e}")
        return None

def sign_request(request: requests.Request) -> requests.PreparedRequest:
    prepared_request = request.prepare()
    now = int(time.time())
    method = request.method.upper()
    path_url = prepared_request.path_url  # includes encoded query params
    body = b'' if prepared_request.body is None else prepared_request.body
    if isinstance(body, str):
        body = body.encode('utf-8')
    data_to_sign = str(now).encode('utf-8') + method.encode('utf-8') + path_url.encode('utf-8') + body
    signature = hmac.new(
        SUMSUB_SECRET_KEY.encode('utf-8'),
        data_to_sign,
        digestmod=hashlib.sha256
    )
    prepared_request.headers['X-App-Token'] = SUMSUB_APP_TOKEN
    prepared_request.headers['X-App-Access-Ts'] = str(now)
    prepared_request.headers['X-App-Access-Sig'] = signature.hexdigest()

    logging.debug(f"Signed Request Headers: {prepared_request.headers}")
    logging.debug(f"Signed Request URL: {prepared_request.url}")
    return prepared_request

async def run_bot():
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Command handler for /start
    app.add_handler(CommandHandler("start", start))

    # Message handler to respond when the bot is mentioned with "kyc" or "poa"
    app.add_handler(MessageHandler(filters.TEXT & filters.ChatType.GROUPS, handle_message))

    # Run the bot until the notebook is interrupted
    await app.run_polling()

# Run the bot
await run_bot()


INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getMe "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/deleteWebhook "HTTP/1.1 200 OK"
INFO:telegram.ext.Application:Application started
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getChatMember "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getChatMember "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA/getChatMember "HTTP/1.1 200 OK"
INFO

RuntimeError: Cannot close a running event loop